# 3. Text Processing & Classification Metrics

**Module 1 · Notebook 3 · Pre-NLP**

## The story so far

Two notebooks ago you built a topic model. Last notebook you built a Naive Bayes text classifier and it worked well enough that your manager sent it to production. Now two new questions are landing on your desk:

1. *"How confident is the model when it predicts 5 stars? Is 0.51 really the same as 0.99?"*
2. *"Can you show me which customers look similar to each other, even though we don't have a label for that?"*

Accuracy alone will not answer either of those. You need **probabilities** (Logistic Regression gives you those for free), a proper metrics toolkit (precision, recall, ROC, PR curves), stronger classifiers when LogReg plateaus (**boosting**), and for the second question you need **clustering** algorithms that find structure without any labels at all.

## Learning objectives

By the end of this notebook you will be able to:

1. Derive the logistic regression equation from linear regression (odds -> log-odds -> sigmoid).
2. Build a text -> token-ids -> padded-tensor pipeline in **pure PyTorch/Python** (no TensorFlow).
3. Read a confusion matrix and compute accuracy / precision / recall / F1 by hand.
4. Pick the right metric for a business scenario (spam -> precision-first, cancer -> recall-first).
5. Train `GradientBoostingClassifier` and `AdaBoostClassifier` and explain how they differ from bagging.
6. Apply K-Means, Agglomerative Hierarchical Clustering, and DBSCAN and pick between them.
7. Use the elbow method and silhouette score to choose `k`.
8. Interpret a dendrogram.

## Prerequisites
- Notebooks 1 & 2 of this module.
- Running on **Google Colab** (free CPU tier is fine, no GPU needed).

> Note: the original version of this notebook used the old TensorFlow `StringLookup` layer and the old `pad_sequences` utility. We have **fully migrated to PyTorch-native tokenization**, which is what the rest of the course uses.


## Section 0. Environment Setup

This notebook is built for Google Colab. Run the install cell once per session (it is a no-op when already installed). Then run the imports cell.


In [ ]:
# Run this ONCE per Colab session.
# If you are running locally in a venv with requirements.txt already installed, you can skip this cell.
!pip install -q "scikit-learn>=1.4" "torch>=2.1" "scipy>=1.11" "pandas>=2.0" "matplotlib>=3.7" "seaborn>=0.13"


In [ ]:
# ---- Core ----
import os
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Modeling ----
import torch
from torch.nn.utils.rnn import pad_sequence

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    GradientBoostingClassifier,
    AdaBoostClassifier,
    RandomForestClassifier,
)
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import load_iris, make_moons
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc,
    silhouette_score,
)
from scipy.cluster.hierarchy import dendrogram, linkage

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- Hyperparameters (kept up top for easy tweaking) ----
TEST_SIZE = 0.2
MAX_LEN = 50
PAD_IDX, UNK_IDX = 0, 1
GBM_N_ESTIMATORS = 100
GBM_MAX_DEPTH = 3
KMEANS_K_RANGE = range(2, 11)
DBSCAN_EPS = 0.3
DBSCAN_MIN_SAMPLES = 5

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

print(f"torch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()} (we do not need one for this notebook)")
print("Environment ready.")


## What are we building today?

Three concrete deliverables, in order:

1. **A PyTorch-native tokenization pipeline**: raw text -> vocab dict -> integer ids -> padded tensor. Reusable for the rest of the course.
2. **A metrics-aware Logistic Regression classifier** on Yelp reviews, with a full precision/recall/ROC tour and threshold tuning.
3. **An unsupervised clustering toolkit** (K-Means, Agglomerative, DBSCAN) applied to real data, with an honest discussion of when each one wins.

The notebook finishes with a **churn prediction lab** where you pick the model and the metric yourself.


## Section 1. Logistic Regression: from linear to sigmoid

### Why linear regression fails on classification

Imagine you try to predict "is this review positive?" (0 or 1) with ordinary linear regression `y = b0 + b1*x`. Two problems:

- Outputs are unbounded: the model will happily predict `-0.7` or `1.3`, which mean nothing as probabilities.
- A single extreme point can swing the whole line and change predictions wildly.

We want an output in `[0, 1]` that behaves like a **probability**. The sigmoid function solves exactly that.

### Odds, log-odds, sigmoid

Probability of event:

```
p = favourable / total
```

Odds = probability of event / probability of non-event:

```
odds = p / (1 - p)
```

The log-odds (logit) are unbounded on both ends, which is exactly what we want a linear combination of features to produce:

```
logit(p) = log(p / (1 - p)) = b0 + b1*x
```

Invert that to get the **sigmoid**:

```
p = 1 / (1 + exp(-(b0 + b1*x)))
```

That is the logistic function. Same linear predictor as OLS, but squashed through `sigmoid` so outputs live in `(0, 1)`.


In [ ]:
# Plot the sigmoid to build intuition.
z = np.linspace(-8, 8, 200)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(7, 3.5))
plt.plot(z, sigmoid, linewidth=2)
plt.axhline(0.5, color="red", linestyle="--", alpha=0.6, label="decision boundary (0.5)")
plt.axvline(0.0, color="grey", linestyle=":", alpha=0.6)
plt.title("Sigmoid: squashes any real number into (0, 1)")
plt.xlabel("z = b0 + b1 x")
plt.ylabel("sigmoid(z)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Demo: LogReg on a 2D toy dataset with a visible decision boundary.
from sklearn.datasets import make_classification

X_toy, y_toy = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, random_state=SEED,
)

clf = LogisticRegression().fit(X_toy, y_toy)

# Make a mesh grid over the feature space to visualise the decision boundary.
xx, yy = np.meshgrid(
    np.linspace(X_toy[:, 0].min() - 1, X_toy[:, 0].max() + 1, 200),
    np.linspace(X_toy[:, 1].min() - 1, X_toy[:, 1].max() + 1, 200),
)
grid_probs = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, grid_probs, levels=20, cmap="RdBu_r", alpha=0.7)
plt.colorbar(label="P(y = 1)")
plt.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap="RdBu_r", edgecolor="k")
plt.title("Logistic Regression - decision surface is a smooth probability")
plt.tight_layout()
plt.show()


In [ ]:
# predict_proba vs predict.
# Most students only call .predict(). That throws away the model's confidence.
sample_probs = clf.predict_proba(X_toy[:6])
sample_preds = clf.predict(X_toy[:6])

print("predict_proba (first 6 rows):")
print(np.round(sample_probs, 3))
print("\npredict (first 6 rows):")
print(sample_preds)

# Histogram of all predicted probabilities:
plt.figure(figsize=(6, 3))
plt.hist(clf.predict_proba(X_toy)[:, 1], bins=20, color="steelblue", edgecolor="white")
plt.title("Distribution of P(y = 1) - are we confident or uncertain?")
plt.xlabel("predicted probability")
plt.ylabel("count")
plt.tight_layout()
plt.show()


### Why probabilities matter

- **Thresholding.** `predict()` uses 0.5 by default. For spam you might want 0.9 (precision-first). For cancer screening you might want 0.2 (recall-first). You can only tune this if you have probabilities.
- **Ranking.** Sort customers by `P(churn)` to prioritise retention calls.
- **Calibration.** A model that says "0.9" should be right about 90% of the time. LogReg is naturally well-calibrated; most neural nets are not.

Keep this distinction sharp: **classification is a probability problem, not a label problem**.


## Section 2. Text tokenization pipeline (PyTorch-native)

Every neural model from here on expects integer token ids, not raw strings. Here is the pipeline we'll reuse in notebooks 5, 6, 8, and 11:

```
raw text -> lowercase/clean -> split on whitespace -> vocab lookup -> integer ids -> padded tensor
```

Previously this course used an older TensorFlow-based pipeline: a `StringLookup` layer to map words to ids and a `pad_sequences` utility to make every sequence the same length. We now do the same thing with **pure Python + `torch.nn.utils.rnn.pad_sequence`**. No TF install, no silent graph behavior, no surprises.


In [ ]:
# Step 1: build a vocabulary from a corpus.
# Two special tokens:
#   0 = <pad>  -> used to make all sequences the same length
#   1 = <unk>  -> used for any word the model has never seen

def build_vocab(texts, min_freq=2):
    """Return a {word: id} dict. Words seen fewer than min_freq times are dropped."""
    counter = Counter(w for t in texts for w in t.lower().split())
    vocab = {"<pad>": PAD_IDX, "<unk>": UNK_IDX}
    for word, count in counter.items():
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab


sample_corpus = [
    "the food was great and the service was great",
    "the service was slow and cold",
    "great food great service great value",
    "cold food slow service never again",
    "warm welcome great value friendly staff",
]

vocab = build_vocab(sample_corpus, min_freq=2)
print(f"vocab size: {len(vocab)}")
print("first 15 entries:", dict(list(vocab.items())[:15]))


In [ ]:
# Step 2: a function that converts one string to a list of ids.
# Unknown words fall back to UNK_IDX = 1. This replaces the old StringLookup layer.

def text_to_ids(text, vocab, unk_idx=UNK_IDX):
    return [vocab.get(w, unk_idx) for w in text.lower().split()]


for review in sample_corpus:
    print(f"{review!r:55s}  ->  {text_to_ids(review, vocab)}")


In [ ]:
# Step 3: pad every sequence to the same length so we can batch them into a 2D tensor.
# This replaces the old pad_sequences utility.

sequences = [text_to_ids(t, vocab) for t in sample_corpus]
print("raw sequences (different lengths):")
for s in sequences:
    print(" ", s)

# Truncate to MAX_LEN, then pad. pad_sequence from torch does the heavy lifting.
tensors = [torch.tensor(s[:MAX_LEN], dtype=torch.long) for s in sequences]
padded = pad_sequence(tensors, batch_first=True, padding_value=PAD_IDX)

print("\npadded tensor shape:", tuple(padded.shape))
print(padded)


### Lab 2.1. Tokenize 5 restaurant reviews

Using the three helpers above (`build_vocab`, `text_to_ids`, and `torch.nn.utils.rnn.pad_sequence`):

1. Build a vocabulary with `min_freq=1` (so every word is kept) from the 5 sample reviews below.
2. Convert each review to a list of token ids using `text_to_ids`.
3. Truncate to `MAX_LEN = 8` and pad into a single 2D tensor of shape `(5, 8)`.
4. Assert the shape and print the tensor.

Expected final tensor shape: `(5, 8)`.


In [ ]:
# Lab 2.1. Exercise

reviews = [
    "the pizza was amazing",
    "service was slow but the food was great",
    "cold pasta and rude staff never again",
    "great atmosphere great wine great pizza",
    "mediocre at best",
]
LAB_MAX_LEN = 8

# 1. Build the vocab (min_freq=1 so every word is kept).
my_vocab = None  # YOUR CODE

# 2. Convert each review to ids.
my_ids = None  # YOUR CODE  -> list of lists of ints

# 3. Truncate to LAB_MAX_LEN then pad into a single tensor.
my_tensors = None  # YOUR CODE  -> list of torch.LongTensor
my_padded = None   # YOUR CODE  -> shape (5, LAB_MAX_LEN) if any review is longer than 8, else (5, max_len_found)

# ---- Verification (do not edit) ----
if my_padded is not None:
    print("vocab size:", len(my_vocab))
    print("padded shape:", tuple(my_padded.shape))
    print(my_padded)
    assert my_padded.shape[0] == 5, "Expected 5 rows (one per review)."
    assert my_padded.shape[1] <= LAB_MAX_LEN, f"Expected at most {LAB_MAX_LEN} columns after truncation."
    print("\nLab 2.1 passed.")


## Section 3. Logistic Regression on Yelp reviews

We'll reuse the Yelp dataset from Notebook 2. TF-IDF gives us sparse numeric features; LogReg fits in milliseconds.


In [ ]:
# Load the same Yelp dataset used in Notebook 2.
YELP_URL = "https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1"
yelp = pd.read_csv(YELP_URL)

# Keep only the extremes so this is a clean binary problem: 1-star (negative) vs 5-star (positive).
yelp_bw = yelp[(yelp.stars == 1) | (yelp.stars == 5)].copy()
yelp_bw["label"] = (yelp_bw["stars"] == 5).astype(int)   # 1 = positive, 0 = negative

print(f"rows: {len(yelp_bw)}   positives: {yelp_bw.label.sum()}   negatives: {(yelp_bw.label == 0).sum()}")

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    yelp_bw["text"], yelp_bw["label"],
    test_size=TEST_SIZE, random_state=SEED, stratify=yelp_bw["label"],
)

tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2), stop_words="english")
X_train = tfidf.fit_transform(X_train_txt)
X_test = tfidf.transform(X_test_txt)

print("TF-IDF train matrix:", X_train.shape)


In [ ]:
# Fit a LogReg. max_iter bumped because TF-IDF is high-dimensional.
logreg = LogisticRegression(max_iter=1000, random_state=SEED)
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["negative (1-star)", "positive (5-star)"]))


In [ ]:
# Inspect the most positive and most negative coefficients.
# Each coefficient tells us how much a word shifts the log-odds toward class 1 (positive).
feature_names = np.array(tfidf.get_feature_names_out())
coefs = logreg.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:][::-1]
top_neg_idx = np.argsort(coefs)[:20]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(range(20), coefs[top_pos_idx][::-1], color="seagreen")
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(feature_names[top_pos_idx][::-1])
axes[0].set_title("Top 20 words driving POSITIVE reviews")

axes[1].barh(range(20), coefs[top_neg_idx][::-1], color="firebrick")
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(feature_names[top_neg_idx][::-1])
axes[1].set_title("Top 20 words driving NEGATIVE reviews")

plt.tight_layout()
plt.show()


### Lab 3.1. Find the 3-star "meh" words

So far we trained on extremes (1 vs 5). But real reviews are a spectrum. 3-star reviews are often the most informative: they tell you *what barely passed*.

1. From the full `yelp` DataFrame, create a binary problem: **3-star vs everything else**.
2. TF-IDF + LogReg (same recipe as above).
3. Print the top 10 words whose coefficients point most strongly toward "3-star". What do they look like?

These words are often hedges ("decent", "okay", "average"). Useful for writing style classifiers later.


In [ ]:
# Lab 3.1. Exercise: identify "meh" (3-star) vocabulary

# 1. Build a binary label: 1 if stars == 3, else 0.
yelp_3 = yelp.copy()
yelp_3["label"] = None  # YOUR CODE

# 2. Train/test split (stratified).
X_tr_txt, X_te_txt, y_tr, y_te = (None, None, None, None)  # YOUR CODE

# 3. TF-IDF + LogReg.
tfidf_3 = None     # YOUR CODE
X_tr = None        # YOUR CODE
X_te = None        # YOUR CODE
logreg_3 = None    # YOUR CODE
# Fit the model here.
# YOUR CODE

# 4. Print accuracy and the top 10 words pointing to "3-star".
# YOUR CODE


## Section 4. Metrics deep dive

### Confusion matrix: the single picture that explains every other metric

|                     | Predicted NEG | Predicted POS |
|---------------------|:-------------:|:-------------:|
| **Actual NEG**      | TN            | FP (Type I)   |
| **Actual POS**      | FN (Type II)  | TP            |

Four numbers, four metrics:

- **Accuracy** = (TP + TN) / total  -> overall hit rate
- **Precision** = TP / (TP + FP)    -> "when I say POS, how often am I right?"
- **Recall**   = TP / (TP + FN)    -> "of all the true POSs, how many did I catch?"
- **F1**       = harmonic mean of precision and recall -> one number when you need to compromise

### Business intuition

| Scenario                  | Which metric matters most? | Why |
|---------------------------|----------------------------|-----|
| Spam filter               | **Precision**              | A false positive = real mail in spam folder. Users rage. |
| Cancer screening          | **Recall**                 | A false negative = missed cancer. Cost >> false alarm. |
| Fraud detection           | **Both (F1)**              | Too many false alarms block customers; too few lets fraud through. |
| Churn prediction (today's final lab) | **Recall**      | Missing a churner = lost revenue forever. Bothering a non-churner = cheap retention email. |


In [ ]:
# Confusion matrix on the Yelp LogReg.
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["pred NEG", "pred POS"],
    yticklabels=["actual NEG", "actual POS"],
    ax=ax,
)
ax.set_title("Confusion matrix - Yelp LogReg")
plt.tight_layout()
plt.show()

TN, FP, FN, TP = cm.ravel()
acc = (TP + TN) / cm.sum()
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}")
print(f"accuracy  = {acc:.4f}")
print(f"precision = {precision:.4f}")
print(f"recall    = {recall:.4f}")
print(f"F1        = {f1:.4f}")


In [ ]:
# Precision-Recall curve and ROC curve: they show the same model at every threshold.
prec, rec, pr_thr = precision_recall_curve(y_test, y_proba)
fpr, tpr, roc_thr = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(rec, prec, linewidth=2)
axes[0].set_title("Precision-Recall curve")
axes[0].set_xlabel("recall")
axes[0].set_ylabel("precision")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1.02)

axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey", alpha=0.6, label="random")
axes[1].set_title("ROC curve")
axes[1].set_xlabel("false positive rate")
axes[1].set_ylabel("true positive rate")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


### Lab 4.1. Precision-first threshold

Your product team wants to surface "guaranteed positive" reviews on the landing page. They need **precision >= 0.95**; anything lower and a negative review might sneak through.

1. Using `y_proba` (predicted probabilities on the Yelp test set) and `y_test`, find the **smallest threshold** such that precision on predictions above that threshold is >= 0.95.
2. Report the resulting **recall** at that threshold.
3. Interpret: what did you give up to gain precision?

Hint: you already have `prec, rec, pr_thr` from the `precision_recall_curve` call above. Note that `precision_recall_curve` returns `len(prec) == len(thresholds) + 1` (last point is (recall=0, precision=1) with no threshold).


In [ ]:
# Lab 4.1. Exercise

# 1. Find the smallest threshold where precision >= 0.95.
chosen_thr = None   # YOUR CODE
chosen_precision = None  # YOUR CODE
chosen_recall = None  # YOUR CODE

# ---- Verification ----
if chosen_thr is not None:
    y_pred_custom = (y_proba >= chosen_thr).astype(int)
    print(f"threshold = {chosen_thr:.3f}")
    print(f"precision at this threshold = {chosen_precision:.3f}")
    print(f"recall at this threshold    = {chosen_recall:.3f}")
    assert chosen_precision >= 0.95, "You need precision >= 0.95"
    print("\nLab 4.1 passed.")


## Section 5. Boosting

Recap of ensemble flavors:

| Family  | Each tree is trained on                | Parallel? | Typical library |
|---------|----------------------------------------|-----------|-----------------|
| Bagging (Random Forest) | a random bootstrap sample of the data | yes | `RandomForestClassifier` |
| Boosting (GBM, AdaBoost, XGBoost) | the **errors** of the previous tree  | no  | `GradientBoostingClassifier`, `AdaBoostClassifier`, `xgboost` |

Key idea of boosting: later trees specialize on samples the earlier trees got wrong. That makes boosted models very accurate but also more prone to overfit, so you have to watch `n_estimators` and `learning_rate`.

**GBM vs AdaBoost:**
- **GBM** fits each new tree to the **gradient of the loss** (hence "gradient boosting"). Works with any differentiable loss.
- **AdaBoost** reweights **misclassified samples** so they dominate the next tree. Simpler, older, easier to derive by hand.


In [ ]:
# GradientBoosting on the same Yelp features.
gbm = GradientBoostingClassifier(
    n_estimators=GBM_N_ESTIMATORS, max_depth=GBM_MAX_DEPTH, random_state=SEED,
)
# Note: GBM on a 50k+ TF-IDF feature matrix is SLOW (minutes on CPU). We subsample for speed.
rng = np.random.RandomState(SEED)
sub_idx = rng.choice(X_train.shape[0], size=2000, replace=False)
gbm.fit(X_train[sub_idx].toarray(), y_train.iloc[sub_idx])
gbm_acc = gbm.score(X_test.toarray(), y_test)
print(f"GBM  accuracy: {gbm_acc:.4f}   (trained on 2k sample for speed)")

ada = AdaBoostClassifier(n_estimators=50, random_state=SEED)
ada.fit(X_train[sub_idx].toarray(), y_train.iloc[sub_idx])
ada_acc = ada.score(X_test.toarray(), y_test)
print(f"AdaBoost accuracy: {ada_acc:.4f}")

logreg_acc = accuracy_score(y_test, y_pred)
print(f"LogReg   accuracy: {logreg_acc:.4f}   (from earlier)")


In [ ]:
# Where does GBM start to overfit? Plot train vs test accuracy as n_estimators grows.
# We use the `staged_predict` trick: it yields predictions after each tree is added, for free.
gbm_lc = GradientBoostingClassifier(n_estimators=120, max_depth=3, random_state=SEED)
gbm_lc.fit(X_train[sub_idx].toarray(), y_train.iloc[sub_idx])

test_dense = X_test.toarray()
train_dense = X_train[sub_idx].toarray()
train_scores = [accuracy_score(y_train.iloc[sub_idx], p) for p in gbm_lc.staged_predict(train_dense)]
test_scores = [accuracy_score(y_test, p) for p in gbm_lc.staged_predict(test_dense)]

plt.figure(figsize=(7, 4))
plt.plot(range(1, len(train_scores) + 1), train_scores, label="train", color="steelblue")
plt.plot(range(1, len(test_scores) + 1), test_scores, label="test", color="firebrick")
plt.xlabel("n_estimators")
plt.ylabel("accuracy")
plt.title("GBM learning curve - where does the test curve start to flatten?")
plt.legend()
plt.tight_layout()
plt.show()


### Lab 5.1. Tune GBM with a small grid

Use `GridSearchCV` to search over:

- `max_depth` ∈ {2, 3, 4}
- `learning_rate` ∈ {0.05, 0.1, 0.2}

with 3-fold CV. Keep `n_estimators=50` so it doesn't take forever.

Print the best params and the best CV score. Fit the best estimator on the full training subset and report test accuracy.


In [ ]:
# Lab 5.1. Exercise

param_grid = None   # YOUR CODE

grid = None   # YOUR CODE: GridSearchCV wrapping a GradientBoostingClassifier(n_estimators=50, random_state=SEED)

# Fit on the subsample (for speed).
# YOUR CODE

# Report best params + best CV score + test accuracy of the best estimator.
# YOUR CODE


## Section 6. Clustering (unsupervised)

Everything so far assumed a label. In the real world 99% of data has no label and labeling is expensive. Clustering finds structure anyway.

Three classic algorithms, each with a different assumption:

| Algorithm             | Assumption                      | When it wins                       |
|-----------------------|---------------------------------|------------------------------------|
| **K-Means**           | Clusters are globular and equal size | Lots of data, you know `k` roughly |
| **Agglomerative**     | Pairwise distances + linkage rule | You want a hierarchy / dendrogram  |
| **DBSCAN**            | Dense regions separated by sparse ones | Non-convex shapes, noisy data      |


In [ ]:
# K-Means on Iris (pretend we don't have the labels).
iris = load_iris()
X_iris = iris.data

inertias = []
silhouettes = []
for k in KMEANS_K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = km.fit_predict(X_iris)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_iris, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(KMEANS_K_RANGE), inertias, marker="o")
axes[0].set_title("Elbow method (inertia vs k)")
axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia")

axes[1].plot(list(KMEANS_K_RANGE), silhouettes, marker="o", color="darkorange")
axes[1].set_title("Silhouette score vs k (higher = better)")
axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")

plt.tight_layout()
plt.show()

best_k = list(KMEANS_K_RANGE)[int(np.argmax(silhouettes))]
print(f"k with best silhouette: {best_k}  (Iris has 3 true classes)")


In [ ]:
# Agglomerative Hierarchical Clustering with a dendrogram.
# Dendrograms become unreadable above ~50 points, so we subsample and use truncate_mode.
rng = np.random.RandomState(SEED)
sub_idx_iris = rng.choice(len(X_iris), size=30, replace=False)
X_small = X_iris[sub_idx_iris]

Z = linkage(X_small, method="ward")

plt.figure(figsize=(10, 4))
dendrogram(Z, truncate_mode="lastp", p=10, leaf_rotation=45.0, show_contracted=True)
plt.title("Dendrogram (30 Iris samples, truncated to last 10 merges)")
plt.xlabel("sample index or (cluster size)")
plt.ylabel("distance")
plt.tight_layout()
plt.show()

# Fit the model with n_clusters=3 for predictions.
agg = AgglomerativeClustering(n_clusters=3)
agg_labels = agg.fit_predict(X_iris)
print("Agglomerative cluster sizes:", np.bincount(agg_labels))


In [ ]:
# Two moons: the textbook dataset where K-Means fails and DBSCAN succeeds.
X_moons, y_moons = make_moons(n_samples=500, noise=0.08, random_state=SEED)

km_moons = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit_predict(X_moons)
db_moons = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES).fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm", s=12)
axes[0].set_title("Ground truth (unknown in practice)")

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=km_moons, cmap="coolwarm", s=12)
axes[1].set_title("K-Means (fails - assumes globular clusters)")

axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=db_moons, cmap="coolwarm", s=12)
axes[2].set_title(f"DBSCAN  eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

n_clusters_ = len(set(db_moons)) - (1 if -1 in db_moons else 0)
print(f"DBSCAN found {n_clusters_} clusters and flagged {(db_moons == -1).sum()} noise points.")


### Lab 6.1. Cluster Yelp reviews into 5 topics (no labels)

Use the TF-IDF features from Section 3 and K-Means with `k=5`. For each cluster, print the **top 10 words** with the largest average TF-IDF. Do they form coherent topics (food, service, decor, value, speed)?

Tip: the Yelp TF-IDF matrix is huge; subsample ~2000 rows for speed. Use `subsample.toarray()` to work in dense for the groupby.


In [ ]:
# Lab 6.1. Exercise: cluster Yelp reviews without labels.

rng = np.random.RandomState(SEED)
sub_idx_yelp = rng.choice(X_train.shape[0], size=2000, replace=False)
X_yelp_sub = X_train[sub_idx_yelp].toarray()
feature_names = np.array(tfidf.get_feature_names_out())

# 1. Fit K-Means with k=5.
km_text = None   # YOUR CODE
cluster_ids = None  # YOUR CODE

# 2. For each cluster, compute mean TF-IDF across its documents and print the top 10 words.
# YOUR CODE


## Section 7. Final lab: Churn prediction (homework)

You are now the data scientist at a telco. Churn costs ~10x more than a false retention call. Your job:

1. Load the churn dataset (Dropbox URL below).
2. Handle missing values (drop? impute? your call, and justify it).
3. Encode categorical columns and scale numeric ones.
4. Train **three** models: Logistic Regression, Random Forest, Gradient Boosting.
5. Compare them on **recall** (primary) and **F1** (secondary). Accuracy is a trap here because non-churners dominate.
6. Pick a winner. Explain in one paragraph *which* model you would ship and *why*, citing the business cost asymmetry.

**Rubric:** full pipeline -> metrics table -> recommendation. ~30-45 minutes of work.


In [ ]:
# Lab 7.1. Exercise: churn prediction end-to-end

CHURN_URL = "https://www.dropbox.com/scl/fi/r8ctv87bqnb42qjmoyqzy/churn_missing.csv?rlkey=30egvb2y4rpsicz3n0t57w6bd&dl=1"
churn = pd.read_csv(CHURN_URL)
print(churn.shape)
churn.head()


In [ ]:
# Your pipeline goes here.

# 1. Inspect + handle missing values.
# YOUR CODE

# 2. Split features X and target y (the 'churn' / 'Churn' column).
X_churn = None   # YOUR CODE
y_churn = None   # YOUR CODE

# 3. Preprocess (one-hot encode categoricals, scale numerics). Hint: sklearn.compose.ColumnTransformer.
# YOUR CODE

# 4. Train/test split (stratified by y_churn).
# YOUR CODE

# 5. Fit three models: LogisticRegression, RandomForestClassifier, GradientBoostingClassifier.
#    Use class_weight='balanced' where available.
# YOUR CODE

# 6. Build a comparison table with accuracy / precision / recall / F1 on the test set.
# YOUR CODE

# 7. Print your recommendation. Which model wins on RECALL?
# YOUR CODE


## Wrap-up and what's next

You now have:

- A **PyTorch-native** tokenization pipeline (vocab dict + `torch.nn.utils.rnn.pad_sequence`). No TensorFlow anywhere in the course from here on.
- A metrics toolkit: confusion matrix, precision, recall, F1, PR curve, ROC curve, threshold tuning.
- Three classifiers compared: LogReg, GBM, AdaBoost (plus RandomForest in the final lab).
- Three clustering algorithms compared: K-Means, Agglomerative, DBSCAN.
- A working churn pipeline with end-to-end preprocessing + model comparison.

### Self-check
1. Your fraud model has precision 0.99 and recall 0.10. Is this good? For what business?
2. If the K-Means elbow plot shows no clear elbow, what does that say about your data?
3. Why can DBSCAN find non-convex clusters that K-Means misses?

### Next up: Module 2. Text Similarity

In Notebook 4 (Capstone 1) you'll apply everything from Module 1 to a real Twitter customer-support routing problem, and then Module 2 opens with **word and document embeddings**: CBOW, Doc2Vec, and fine-tuned transformers. The tokenization pipeline you built today is the input to every model in Module 2.

This completes Notebook 3 and Module 1.
